# Week 7: Transfer Learning (BERT & SBERT)

Welcome to Week 7! This week, we transition from building our own Transformer blocks to exploring how these blocks are stacked to form BERT, a powerful encoder-only model. We will also dive into transfer learning, fine-tuning, and Sentence-BERT (SBERT) for semantic search.

## Day 1: Building an Encoder

**Goal:** Extend Week 6's Transformer block into a full Encoder architecture. Understand how multiple layers stack to build deep contextual representations.

Let's leverage the `TinyBERT` class we added to `utils.py`.

In [ ]:
import torch
from utils import create_bert

# Initialize our TinyBERT encoder from scratch
vocab_size = 30000
d_model = 256
num_heads = 8
mlp_hidden_dim = 1024
num_layers = 4
pad_id = 0

tiny_bert = create_bert(
    vocab_size=vocab_size,
    d_model=d_model,
    num_heads=num_heads,
    mlp_hidden_dim=mlp_hidden_dim,
    num_layers=num_layers,
    pad_id=pad_id,
    use_rope=True,
    use_absolute_positions=False
)

print(f"TinyBERT Parameters: {sum(p.numel() for p in tiny_bert.parameters())}")
print(tiny_bert)

## Day 2: BERT Architecture & Pre-training

**Goal:** Introduce BERT (Bidirectional Encoder Representations from Transformers). Explore its Masked Language Modeling (MLM) and Next Sentence Prediction (NSP) objectives.

In [ ]:
# Simulating the Masked Language Modeling (MLM) logic
def create_mlm_labels(input_ids, mask_token_id, pad_token_id, mask_prob=0.15):
    """
    A simplified version of MLM masking.
    """
    labels = input_ids.clone()
    
    # Create a random probability matrix for tokens
    prob_matrix = torch.rand(input_ids.shape)
    
    # Don't mask padding tokens
    prob_matrix.masked_fill_(input_ids == pad_token_id, 1.0)
    
    # Mask tokens where prob < mask_prob
    masked_indices = prob_matrix < mask_prob
    
    # Set masked inputs to mask_token_id
    inputs_masked = input_ids.clone()
    inputs_masked[masked_indices] = mask_token_id
    
    # We only compute loss on masked tokens, set others to -100 (ignored by CrossEntropyLoss)
    labels[~masked_indices] = -100 
    
    return inputs_masked, labels

dummy_inputs = torch.randint(1, vocab_size, (2, 20))
masked_inputs, labels = create_mlm_labels(dummy_inputs, mask_token_id=1, pad_token_id=0)
print("Original:", dummy_inputs[0][:10])
print("Masked:  ", masked_inputs[0][:10])
print("Labels:  ", labels[0][:10])

## Day 3: Fine-tuning BERT

**Goal:** Hands-on with fine-tuning a pre-trained BERT model for a classification task using the Hugging Face `Trainer` API.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset

# 1. Load the IMDB dataset (our anchor dataset)
# dataset = load_dataset("imdb")

# 2. Load pre-trained BERT tokenizer and model for sequence classification
model_name = "prajjwal1/bert-tiny" # Using a tiny BERT for speed
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 3. Example tokenization
sample_text = "This movie was absolutely fantastic!"
tokens = tokenizer(sample_text, return_tensors="pt")
print("Tokens:", tokens)

# 4. Forward pass
outputs = model(**tokens)
print("Logits:", outputs.logits)

## Day 4: Sentence-BERT (SBERT) and Siamese Networks

**Goal:** Introduce SBERT and Siamese networks. Understand why standard BERT is slow for semantic similarity and how SBERT solves this by producing sentence embeddings.

In [ ]:
from utils import create_sbert

# We can wrap our previously created TinyBERT in our SBERT wrapper
# This will apply Mean Pooling to the BERT outputs and L2 normalize them.
tiny_sbert = create_sbert(tiny_bert)

dummy_attention_mask = torch.ones_like(dummy_inputs)
sentence_embeddings = tiny_sbert(dummy_inputs, dummy_attention_mask)

print(f"Sentence embeddings shape: {sentence_embeddings.shape} (batch_size, d_model)")
print(f"Vector norm (should be 1.0 due to L2 normalization): {torch.norm(sentence_embeddings[0]).item():.4f}")

## Day 5: SBERT in Action (Semantic Search)

**Goal:** Implement semantic search or retrieval using SBERT embeddings and compare performance with Week 3's baseline.

In [ ]:
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer

# Load a pre-trained SBERT model
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

sentences = [
    "The weather is lovely today.",
    "It's so sunny outside!",
    "He drove a fast car."
]

# Generate embeddings
embeddings = sbert_model.encode(sentences, convert_to_tensor=True)

# Compute Cosine Similarities
sim_0_1 = F.cosine_similarity(embeddings[0].unsqueeze(0), embeddings[1].unsqueeze(0))
sim_0_2 = F.cosine_similarity(embeddings[0].unsqueeze(0), embeddings[2].unsqueeze(0))

print(f"Similarity ('{sentences[0]}' vs '{sentences[1]}'): {sim_0_1.item():.4f}")
print(f"Similarity ('{sentences[0]}' vs '{sentences[2]}'): {sim_0_2.item():.4f}")